# Medallion Architecture (File-Based Pipeline)
Bronze → Silver → Gold using Folder Structure

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score


## Step 1: Create Folder Structure

In [ ]:
os.makedirs('data/bronze', exist_ok=True)
os.makedirs('data/silver', exist_ok=True)
os.makedirs('data/gold', exist_ok=True)
os.makedirs('data/feature_store', exist_ok=True)

print("Folders created")

## Step 2: Bronze Layer (Raw Data)

In [ ]:
df = pd.read_csv('telco_customer_churn.csv')  # Replace path

df.to_csv('data/bronze/bronze_raw.csv', index=False)

print("Bronze data stored")

## Step 3: Silver Layer (Cleaning)

In [ ]:
bronze_df = pd.read_csv('data/bronze/bronze_raw.csv')

silver_df = bronze_df.copy()
silver_df = silver_df.drop_duplicates()
silver_df.fillna(method='ffill', inplace=True)

silver_df.to_csv('data/silver/silver_cleaned.csv', index=False)

print("Silver data created")

## Step 4: Data Quality Validation

In [ ]:
silver_df = pd.read_csv('data/silver/silver_cleaned.csv')

print("Schema:\n", silver_df.dtypes)
print("\nNull Values:\n", silver_df.isnull().sum())

numeric_cols = silver_df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    outliers = ((silver_df[col] > silver_df[col].mean() + 3*silver_df[col].std()) | 
                (silver_df[col] < silver_df[col].mean() - 3*silver_df[col].std())).sum()
    print(f"{col} Outliers: {outliers}")

## Step 5: Feature Engineering (Gold Layer)

In [ ]:
silver_df = pd.read_csv('data/silver/silver_cleaned.csv')

gold_df = silver_df.copy()

if 'tenure' in gold_df.columns:
    gold_df['TenureGroup'] = pd.cut(gold_df['tenure'], 
                                   bins=[0,12,24,48,60,100], 
                                   labels=['0-12','12-24','24-48','48-60','60+'])

if 'MonthlyCharges' in gold_df.columns and 'tenure' in gold_df.columns:
    gold_df['MonthlyCharges_Tenure_Ratio'] = gold_df['MonthlyCharges'] / (gold_df['tenure'] + 1)

gold_df.to_csv('data/gold/gold_features.csv', index=False)

print("Gold data created")

## Step 6: Feature Store Simulation

In [ ]:
gold_df = pd.read_csv('data/gold/gold_features.csv')

feature_store = gold_df[['TenureGroup']].copy()
feature_store['Last_Updated'] = datetime.now().date()

feature_store.to_csv('data/feature_store/feature_store.csv', index=False)

print("Feature store created")

## Step 7: ML Model

In [ ]:
gold_df = pd.read_csv('data/gold/gold_features.csv')

if 'Churn' in gold_df.columns:
    df_model = gold_df.copy()

    for col in df_model.select_dtypes(include='object').columns:
        df_model[col] = LabelEncoder().fit_transform(df_model[col].astype(str))

    X = df_model.drop('Churn', axis=1)
    y = df_model['Churn']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))